# 🚦 AmpelPilot - 멀티 데이터셋 자동 병합 & 고성능 YOLOv8n 학습 & TFLite 변환

이 노트북은 여러 Roboflow 보행자 신호등 데이터셋을 하나로 자동 병합하여
**우리가 원하는 딱 3가지 핵심 클래스**로 정규화해 학습시킵니다:

1. 🔴 **`red`** : 보행자 빨간불 (red, redlight, stop, dont_walk 등 통합)
2. 🟢 **`green`** : 보행자 초록불 (green, greenlight, walk, go 등 통합)
3. ⬛ **`pedestrian Traffic Light`** : 보행자 신호등 기둥/하우징 외형 (불 꺼진 신호등 감지용!)

---

### ⚡ 시작 전 확인
상단 메뉴 **[런타임] → [런타임 유형 변경] → 하드웨어 가속기: `T4 GPU`** 로 설정되어 있는지 확인하세요!

In [ ]:
# ============================================================
# [셀 1] Roboflow API 키 입력
# 무료 발급: https://app.roboflow.com 접속 ➔ Settings ➔ API Keys
# ============================================================
ROBOFLOW_API_KEY = "여기에_본인_API_키_입력"  # <--- 본인 키로 변경!
# ============================================================

In [ ]:
# [셀 2] 필수 라이브러리 설치 (10초)
!pip install -q ultralytics roboflow
print("✅ 패키지 설치 완료!")

In [ ]:
# [셀 3] 멀티 데이터셋 다운로드 & '3개 클래스'로 자동 병합 및 정규화
from roboflow import Roboflow
import os, glob, shutil, yaml

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# -------------------------------------------------------------
# 📦 병합할 데이터셋 목록 (사용자 지정 + 고품질 추천 데이터셋)
# -------------------------------------------------------------
DATASET_CONFIGS = [
    {
        "name": "s-workspace-cosh1",
        "workspace": "s-workspace-cosh1",
        "project": "pedestrian-traffic-light-pbbl6",
        "version": 1,
        "desc": "사용자 추천: greenlight / redlight 데이터셋"
    },
    {
        "name": "ono-gedd7",
        "workspace": "ono-gedd7",
        "project": "pedestrian-traffic-light-puf4a",
        "version": 1,
        "desc": "유럽형 보행자 신호등 하우징 및 red/green"
    },
    {
        "name": "cible",
        "workspace": "cible",
        "project": "pedestrian-traffic-light",
        "version": 1,
        "desc": "보행자 신호등 전문 1,000+ 대용량 데이터셋"
    }
]

# 통합 데이터셋 저장소
UNIFIED_DIR = "/content/unified_dataset"
if os.path.exists(UNIFIED_DIR):
    shutil.rmtree(UNIFIED_DIR)

for split in ["train", "valid"]:
    os.makedirs(os.path.join(UNIFIED_DIR, "images", split), exist_ok=True)
    os.makedirs(os.path.join(UNIFIED_DIR, "labels", split), exist_ok=True)

# -------------------------------------------------------------
# 🎯 3개 표준 클래스 매핑 함수
# 0: red, 1: green, 2: pedestrian Traffic Light
# -------------------------------------------------------------
TARGET_CLASSES = ["red", "green", "pedestrian Traffic Light"]

def map_label_name(raw_name):
    n = raw_name.strip().lower().replace(" ", "").replace("_", "").replace("-", "")
    # 0: Red
    if any(k in n for k in ["redlight", "red", "stop", "dontwalk"]):
        return 0, "red"
    # 1: Green
    elif any(k in n for k in ["greenlight", "green", "walk", "go"]):
        return 1, "green"
    # 2: Pedestrian Traffic Light Housing
    elif any(k in n for k in ["pedestrian", "trafficlight", "housing", "signal", "ampel"]):
        return 2, "pedestrian Traffic Light"
    return None, None

# -------------------------------------------------------------
# 📥 다운로드 및 병합 루프 실행
# -------------------------------------------------------------
total_images_copied = 0
total_labels_kept = 0
class_counts = {0: 0, 1: 0, 2: 0}

for ds_idx, cfg in enumerate(DATASET_CONFIGS):
    print("=" * 60)
    print(f"[{ds_idx+1}/{len(DATASET_CONFIGS)}] {cfg['workspace']}/{cfg['project']} ({cfg['desc']}) 다운로드 중...")
    try:
        proj = rf.workspace(cfg["workspace"]).project(cfg["project"])
        try:
            ds = proj.version(cfg.get("version", 1)).download("yolov8")
        except Exception:
            try:
                ds = proj.version(proj.versions[0].version).download("yolov8")
            except Exception:
                ds = proj.version(4).download("yolov8")
    except Exception as e:
        print(f"⚠️ {cfg['workspace']}/{cfg['project']} 다운로드 건너뜀 (사유: {e})")
        continue

    # data.yaml 파싱
    yaml_file = os.path.join(ds.location, "data.yaml")
    if not os.path.exists(yaml_file):
        continue
    with open(yaml_file, "r") as f:
        ds_cfg = yaml.safe_load(f)

    orig_names = ds_cfg.get("names", [])
    if isinstance(orig_names, dict):
        orig_names = [orig_names[k] for k in sorted(orig_names.keys())]

    # 클래스 ID 매핑표 작성
    id_map = {}
    for orig_id, name in enumerate(orig_names):
        mapped_id, mapped_name = map_label_name(name)
        if mapped_id is not None:
            id_map[orig_id] = mapped_id
    print(f"   🔎 원본 클래스: {orig_names}")
    print(f"   🎯 표준 매핑: {id_map}")

    # 분할별 파일 복사 및 라벨 변환
    for split in ["train", "valid"]:
        in_img_dir = os.path.join(ds.location, split, "images")
        in_lbl_dir = os.path.join(ds.location, split, "labels")
        if not os.path.isdir(in_img_dir):
            continue

        out_img_dir = os.path.join(UNIFIED_DIR, "images", split)
        out_lbl_dir = os.path.join(UNIFIED_DIR, "labels", split)

        img_exts = (".jpg", ".jpeg", ".png", ".bmp")
        for img_path in glob.glob(os.path.join(in_img_dir, "*.*")):
            if not img_path.lower().endswith(img_exts):
                continue
            base = os.path.splitext(os.path.basename(img_path))[0]
            lbl_path = os.path.join(in_lbl_dir, base + ".txt")

            # 라벨 필터링
            new_lines = []
            if os.path.exists(lbl_path):
                with open(lbl_path, "r") as lf:
                    for line in lf:
                        parts = line.strip().split()
                        if not parts: continue
                        orig_cid = int(parts[0])
                        if orig_cid in id_map:
                            target_cid = id_map[orig_cid]
                            new_lines.append(f"{target_cid} {' '.join(parts[1:])}\n")
                            class_counts[target_cid] += 1
                            total_labels_kept += 1

            # 유효한 라벨이 있는 경우에만 복사 (공백 이미지 제외하여 데이터 품질 극대화)
            if new_lines:
                unique_name = f"ds{ds_idx}_{base}"
                shutil.copy2(img_path, os.path.join(out_img_dir, unique_name + os.path.splitext(img_path)[1]))
                with open(os.path.join(out_lbl_dir, unique_name + ".txt"), "w") as lf:
                    lf.writelines(new_lines)
                total_images_copied += 1

# -------------------------------------------------------------
# 📄 통합 data.yaml 생성
# -------------------------------------------------------------
unified_yaml_path = os.path.join(UNIFIED_DIR, "data.yaml")
unified_yaml_content = {
    "path": UNIFIED_DIR,
    "train": "images/train",
    "val": "images/valid",
    "names": {0: "red", 1: "green", 2: "pedestrian Traffic Light"},
    "nc": 3
}

with open(unified_yaml_path, "w") as f:
    yaml.safe_dump(unified_yaml_content, f)

print("=" * 60)
print(f"🎉 모든 데이터셋 병합 완료!")
print(f"   총 복사된 이미지: {total_images_copied} 장")
print(f"   총 유효 라벨 수: {total_labels_kept} 개")
print(f"   - 🔴 [0] red : {class_counts[0]} 개")
print(f"   - 🟢 [1] green : {class_counts[1]} 개")
print(f"   - ⬛ [2] pedestrian Traffic Light : {class_counts[2]} 개")
print(f"   통합 data.yaml: {unified_yaml_path}")
print("=" * 60)


In [ ]:
# [셀 4] 대용량 통합 데이터셋 + 데이터 증강(Augmentation) 강화 YOLOv8n 학습 (약 15~20분)
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=unified_yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    name='ampelpilot_3class_unified',
    patience=12,
    # --- 강력한 실전 데이터 증강(Augmentation) 옵션 ---
    scale=0.5,        # 0.5배~1.5배 원거리/근거리 신호등 크기 적응
    degrees=12.0,     # 스마트폰 기울임 각도 대응
    hsv_h=0.015,      # 색조 미세 조정
    hsv_s=0.7,        # 채도 변화 (흐린 날, 비 오는 날 대응)
    hsv_v=0.4,        # 밝기 변화 (밤, 역광, 모니터 화면 불빛 대응)
    mosaic=1.0,       # 4장 사진 합성으로 다양한 배경 학습
    mixup=0.1,        # 이미지 믹스업
    save=True,
    plots=True
)

print("\n🎉 고성능 전이학습 완료!")


In [ ]:
# [셀 5] 학습 결과 그래프 및 샘플 예측 확인
from IPython.display import Image, display

res_img = '/content/runs/detect/ampelpilot_3class_unified/results.png'
if os.path.exists(res_img):
    print("📊 학습 결과 곡선:")
    display(Image(filename=res_img, width=800))

preds = glob.glob('/content/runs/detect/ampelpilot_3class_unified/val_batch*_pred.jpg')
if preds:
    print("🔍 실제 신호등 감지 결과 샘플:")
    display(Image(filename=preds[0], width=600))


In [ ]:
# [셀 6] 안드로이드용 TFLite (Float16) 변환 (약 6MB)
best_weight = '/content/runs/detect/ampelpilot_3class_unified/weights/best.pt'
model_best = YOLO(best_weight)

print("⚙️ TFLite 모델로 변환 중...")
exported = model_best.export(
    format='tflite',
    imgsz=640,
    half=True  # Float16 양자화 (용량 절반 & 모바일 속도 가속)
)

tflite_files = glob.glob('/content/runs/detect/ampelpilot_3class_unified/weights/**/*.tflite', recursive=True)
if not tflite_files:
    tflite_files = glob.glob('/content/**/*.tflite', recursive=True)

for f in tflite_files:
    size_mb = os.path.getsize(f) / (1024*1024)
    print(f"  📦 생성 완료: {f} ({size_mb:.2f} MB)")


In [ ]:
# [셀 7] labelmap.txt 생성 & 자동 다운로드
from google.colab import files

labelmap_path = '/content/labelmap.txt'
with open(labelmap_path, 'w') as f:
    for name in target_classes:
        f.write(name + '\n')

print(f"📄 최종 labelmap.txt 생성 완료:")
with open(labelmap_path, 'r') as f:
    print(f.read())

print("📥 파일 다운로드 시작...")
if tflite_files:
    files.download(tflite_files[0])
files.download(labelmap_path)

print("\n" + "="*60)
print("🎉 완벽합니다! 다운로드된 두 파일을 아래 위치에 넣으세요:")
print("   AmpelPilot/app/src/main/assets/")
print("   1) *.tflite 파일 ➔ detect.tflite 로 이름 변경")
print("   2) labelmap.txt ➔ 그대로 덮어쓰기")
print("="*60)